In [ ]:
!pip install agentic_vnstock vnstock

In [ ]:
import pandas as pd
from agentic_vnstock.client import AgenticVNStock

# Khởi tạo client
client = AgenticVNStock()

In [ ]:
import os
import pandas as pd
import vnstock # Import the vnstock library
from agentic_vnstock.client import AgenticVNStock

# Initialize the client
client = AgenticVNStock()

In [ ]:
# 1.1 Giá lịch sử (Sử dụng API Entrade/DNSE)
df_history = client.get_stock_historical('FPT', '2024-01-01', '2024-01-10')
display("Giá lịch sử:", df_history.head())

In [ ]:
import vnstock

# Load the full stock symbol listing for the Vietnamese stock market
all_companies_df = vnstock.Listing().all_symbols()
print(f"Loaded {len(all_companies_df)} stock symbols from vnstock.Listing().all_symbols()")
display(all_companies_df.head(10))

# Keep only the symbol and company name columns for later processing
if "symbol" in all_companies_df.columns:
    all_companies_df = all_companies_df[["symbol", "organ_name"]]


In [ ]:
import os
import time
import pandas as pd

# Extract stock symbols from the DataFrame
try:
    if isinstance(all_companies_df, pd.DataFrame):
        if not all_companies_df.empty and "symbol" in all_companies_df.columns:
            stock_symbols = all_companies_df["symbol"].dropna().astype(str).tolist()
            print(f"Successfully retrieved {len(stock_symbols)} stock symbols from all_companies_df.")
        elif not all_companies_df.empty and "ticker" in all_companies_df.columns:
            stock_symbols = all_companies_df["ticker"].dropna().astype(str).tolist()
            print(f"Successfully retrieved {len(stock_symbols)} stock symbols using 'ticker' column from all_companies_df.")
        else:
            print("Could not retrieve stock symbols. Listing DataFrame does not contain 'symbol' or 'ticker' column, or is empty.")
            stock_symbols = []
    else:
        print("all_companies_df exists but is not a pandas DataFrame.")
        stock_symbols = []
except NameError:
    print("all_companies_df is not defined. Define or load it before running this cell.")
    stock_symbols = []

print(f"Stock symbols to process: {stock_symbols[:20]}... (showing first 20)")
print(f"Total symbols: {len(stock_symbols)}")

start_date = "2021-01-01"
end_date = "2026-05-31"
output_dir = "/content/drive/MyDrive/Colab Notebooks/PT&QLDT_Nang_cao"
output_filename = "historical_stock_data_full.csv"
output_filepath = os.path.join(output_dir, output_filename)

os.makedirs(output_dir, exist_ok=True)

all_ohlcv_data = []
failed_symbols = []
skipped_symbols = []

for idx, symbol in enumerate(stock_symbols, start=1):
    if not symbol or not isinstance(symbol, str):
        skipped_symbols.append(symbol)
        continue

    print(f"[{idx}/{len(stock_symbols)}] Fetching historical data for {symbol}...")
    try:
        df_symbol = client.get_stock_historical(symbol, start_date, end_date)
        if isinstance(df_symbol, pd.DataFrame) and not df_symbol.empty:
            df_symbol = df_symbol.copy()
            df_symbol["symbol"] = symbol
            all_ohlcv_data.append(df_symbol)
        else:
            print(f"  No historical data for {symbol}.")
            failed_symbols.append(symbol)
    except Exception as e:
        print(f"  Error fetching {symbol}: {type(e).__name__}: {e}")
        failed_symbols.append(symbol)
    time.sleep(0.5)  # Avoid rapid-fire API calls

if all_ohlcv_data:
    combined_df = pd.concat(all_ohlcv_data, ignore_index=True)
    combined_df.to_csv(output_filepath, index=False)
    print(f"Saved {len(combined_df)} rows of OHLCV data to {output_filepath}")
    display(combined_df.head())
    display(combined_df.tail())
else:
    print("No historical data was fetched for any of the symbols.")

if failed_symbols:
    print(f"Failed symbols: {len(failed_symbols)}")
    print(failed_symbols[:20])
    with open(os.path.join(output_dir, "failed_stock_symbols.txt"), "w", encoding="utf-8") as f:
        f.write("
".join(failed_symbols))
    print(f"Failed symbols saved to {os.path.join(output_dir, 'failed_stock_symbols.txt')}")

if skipped_symbols:
    print(f"Skipped symbols count: {len(skipped_symbols)}")
